# Prep smoke test: Harvey LAB RAG on a Colab T4

**Goal:** before the timed build, check that every risky component works on a free Colab T4, and measure how fast it is. These numbers decide the open design choices (which small LLM, what eval size, concurrency).

| Step | What it checks | Decides |
|---|---|---|
| 1 | Download the `antitrust-competition` subset, quick-parse it, corpus stats | chunk counts, token budget, parser edge cases |
| 2 | `claude_agent_sdk` with subscription OAuth from Colab Secrets: latency, structured output, concurrency | LLM-call budget, eval size |
| 3 | BGE-M3 embeddings, FAISS, BM25, bge-reranker-v2-m3 on the GPU | index build time, rerank and CRAG strip-scoring cost |
| 4 | Phoenix inside Colab and a traced toy LangGraph that calls Claude | observability approach |
| 5 | Small-LLM chunk contextualization: HF transformers vs. vLLM (isolated venv) | Contextual Retrieval: local LLM, or the deterministic fallback |

**Before you run:**
1. *Runtime → Change runtime type → T4 GPU.*
2. On your laptop, run `claude setup-token` and copy the token.
3. In Colab, open 🔑 **Secrets** and add `CLAUDE_CODE_OAUTH_TOKEN` (turn on *Notebook access*). `ANTHROPIC_API_KEY` is optional; it tests the fallback path.
4. Run the cells top to bottom. If Colab asks you to **restart the session** after the install cell, restart it and continue from the next cell.
5. At the end, copy the printed `RESULTS` JSON back to Claude.

The notebook never prints secrets. Even so, don't share this notebook with outputs saved if you edit it to print anything sensitive.

## 0. Install (everything except vLLM, which gets its own venv in step 5)

In [ ]:
import time, subprocess
_t0 = time.perf_counter()
!pip -q install python-docx openpyxl python-pptx claude-agent-sdk anthropic faiss-cpu bm25s \
    arize-phoenix arize-phoenix-otel openinference-instrumentation-langchain langgraph
INSTALL_S = round(time.perf_counter() - _t0, 1)
print("install seconds:", INSTALL_S)

In [ ]:
import json, os, platform, sys, time, gc, collections, statistics, subprocess
# Colab preinstalls GPU JAX, and bm25s runs jax.lax.top_k at import time. JAX then pre-allocates 75% of the T4
# outside torch's allocator (we hit this: 11.3 GB held, torch saw 0.01 GB). Keep JAX on the CPU.
# This must be set before anything imports jax.
os.environ["JAX_PLATFORMS"] = "cpu"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
from importlib.metadata import version
import torch

RESULTS = {}
def record(section, **kv):
    RESULTS.setdefault(section, {}).update(kv)
    print(f"[{section}]", json.dumps(kv, default=str))

def v(p):
    try: return version(p)
    except Exception: return None

def peak_gb():
    return round(torch.cuda.max_memory_allocated() / 2**30, 2)

def gpu_report(tag):
    # torch's OOM message counts ALL used GPU memory as "this process"; this separates torch's share from the rest
    free, total = torch.cuda.mem_get_info()
    procs = subprocess.run(["nvidia-smi", "--query-compute-apps=pid,process_name,used_memory", "--format=csv,noheader"],
                           capture_output=True, text=True).stdout.strip()
    info = dict(free_gb=round(free / 2**30, 2), total_gb=round(total / 2**30, 2),
                torch_allocated_gb=round(torch.cuda.memory_allocated() / 2**30, 2),
                torch_reserved_gb=round(torch.cuda.memory_reserved() / 2**30, 2),
                gpu_procs=procs or "(none visible: containers often hide PIDs)")
    RESULTS.setdefault("gpu", {})[tag] = info
    print(f"[gpu:{tag}]", info)
    return info

# transformers v5 renamed torch_dtype -> dtype
DTYPE_KW = "dtype" if int((v("transformers") or "5").split(".")[0]) >= 5 else "torch_dtype"

gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
record("env",
       python=platform.python_version(), gpu=gpu,
       capability=torch.cuda.get_device_capability(0) if gpu else None,
       vram_gb=round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1) if gpu else None,
       ram_gb=round(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 2**30, 1),
       cuda=torch.version.cuda, install_s=globals().get("INSTALL_S"),
       pkgs={p: v(p) for p in ["torch", "transformers", "sentence-transformers", "claude-agent-sdk",
                               "langgraph", "arize-phoenix", "bm25s", "faiss-cpu"]})
assert gpu, "No GPU: Runtime → Change runtime type → T4 GPU"

## 1. Data: sparse-checkout the antitrust subset, quick parse, corpus stats
This parser is deliberately crude, just enough to get text and numbers. The structure-aware parser comes in the real build. One thing we specifically check here is **tracked changes** (`w:ins` / `w:del`) in the docx files. `python-docx`'s `paragraph.text` silently **drops inserted text** inside `w:ins`, which matters for the redline documents.

In [ ]:
from pathlib import Path
t0 = time.perf_counter()
if not os.path.isdir("/content/harvey-labs"):
    subprocess.run("cd /content && git clone -q --depth 1 --filter=blob:none --sparse "
                   "https://github.com/harveyai/harvey-labs.git && cd harvey-labs && "
                   "git sparse-checkout set tasks/antitrust-competition", shell=True, check=True)
record("data", clone_s=round(time.perf_counter() - t0, 1))
ROOT = Path("/content/harvey-labs/tasks/antitrust-competition")
print(len(list(ROOT.glob("*/task.json"))), "tasks")

In [ ]:
import email, zipfile
from email import policy
from docx import Document
import openpyxl
from pptx import Presentation

def quick_text(p: Path) -> str:
    ext = p.suffix.lower()
    if ext == ".docx":
        d = Document(p)
        parts = [para.text for para in d.paragraphs if para.text.strip()]
        for t in d.tables:
            for r in t.rows:
                parts.append(" | ".join(c.text.strip() for c in r.cells))
        return "\n".join(parts)
    if ext == ".xlsx":
        wb = openpyxl.load_workbook(p, data_only=True, read_only=True)
        out = []
        for ws in wb.worksheets:
            out.append(f"## Sheet: {ws.title}")
            for row in ws.iter_rows(values_only=True):
                if any(x is not None for x in row):
                    out.append(" | ".join("" if x is None else str(x) for x in row))
        return "\n".join(out)
    if ext == ".pptx":
        out = []
        for i, s in enumerate(Presentation(p).slides, 1):
            out.append(f"## Slide {i}")
            out += [sh.text_frame.text for sh in s.shapes if sh.has_text_frame and sh.text_frame.text.strip()]
        return "\n".join(out)
    if ext == ".eml":
        m = email.message_from_bytes(p.read_bytes(), policy=policy.default)
        body = m.get_body(preferencelist=("plain", "html"))
        return (f"From: {m['from']}\nTo: {m['to']}\nDate: {m['date']}\nSubject: {m['subject']}\n\n"
                + (body.get_content() if body else ""))
    return p.read_text(errors="ignore")

docs, errors, tracked = [], [], []
for p in sorted(ROOT.glob("*/documents/*")):
    try:
        docs.append(dict(task=p.parent.parent.name, doc=p.name, ext=p.suffix.lower(), text=quick_text(p)))
    except Exception as e:
        errors.append((p.name, repr(e)[:200]))
    if p.suffix.lower() == ".docx":
        x = zipfile.ZipFile(p).read("word/document.xml").decode("utf8", "ignore")
        n_ins, n_del = x.count("<w:ins "), x.count("<w:del ")
        if n_ins or n_del:
            tracked.append((p.parent.parent.name, p.name, n_ins, n_del))

from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("BAAI/bge-m3")
for d in docs:
    d["n_tok"] = len(tok(d["text"], add_special_tokens=False)["input_ids"])
per_task = collections.Counter()
for d in docs:
    per_task[d["task"]] += d["n_tok"]

record("data", n_tasks=len(per_task), n_docs=len(docs), parse_errors=errors[:5],
       by_ext=dict(collections.Counter(d["ext"] for d in docs)),
       total_tokens=sum(d["n_tok"] for d in docs),
       doc_tokens_median=int(statistics.median(d["n_tok"] for d in docs)),
       doc_tokens_p90=int(sorted(d["n_tok"] for d in docs)[int(0.9 * len(docs))]),
       doc_tokens_max=max(d["n_tok"] for d in docs),
       task_tokens_median=int(statistics.median(per_task.values())),
       task_tokens_max=max(per_task.values()),
       docx_with_tracked_changes=len(tracked), tracked_examples=tracked[:5])

In [ ]:
# Naive 512-token windows, only for the throughput benchmarks below (the real build uses structure-aware chunks).
CHUNK_TOK = 512
chunks = []
for d in docs:
    ids = tok(d["text"], add_special_tokens=False)["input_ids"]
    for i in range(0, len(ids), CHUNK_TOK):
        chunks.append(dict(task=d["task"], doc=d["doc"], text=tok.decode(ids[i:i + CHUNK_TOK])))
record("data", n_chunks_naive512=len(chunks))

## 2. Claude via `claude_agent_sdk` (subscription OAuth)
Things we learn here:
- **Per-call overhead.** Every `query()` starts a Claude Code CLI subprocess. `wall_s - api_s` is that overhead.
- **Structured output.** `output_format` with a JSON schema is what the router and the CRAG grader will use. Does it work with `max_turns=1`?
- **Concurrency.** How many parallel calls go through before rate limiting? This sets how big the eval can be.

`ANTHROPIC_API_KEY` is removed from the subprocess environment on purpose: if it's present, it takes precedence over the OAuth token and silently bills the API instead of your subscription.

In [ ]:
import asyncio
from google.colab import userdata
from claude_agent_sdk import query, ClaudeAgentOptions, ResultMessage

def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

OAUTH = secret("CLAUDE_CODE_OAUTH_TOKEN")
API_KEY = secret("ANTHROPIC_API_KEY")
os.environ.pop("ANTHROPIC_API_KEY", None)
record("sdk", has_oauth_secret=bool(OAUTH), has_api_key_secret=bool(API_KEY))
STDERR = []

def make_opts(model="haiku", max_turns=1, **kw):
    return ClaudeAgentOptions(
        model=model, tools=[], max_turns=max_turns, setting_sources=[],
        system_prompt="You are a precise legal research assistant. Answer concisely.",
        env={"CLAUDE_CODE_OAUTH_TOKEN": OAUTH or "", "ANTHROPIC_API_KEY": ""},
        stderr=STDERR.append, **kw)

async def ask(prompt, **kw):
    t0 = time.perf_counter(); res = None
    async for m in query(prompt=prompt, options=make_opts(**kw)):
        if isinstance(m, ResultMessage):
            res = m
    wall = round(time.perf_counter() - t0, 2)
    if res is None:
        return None, dict(wall_s=wall, error="no ResultMessage")
    return res.result, dict(
        wall_s=wall, api_s=round(res.duration_api_ms / 1000, 2), is_error=res.is_error,
        subtype=res.subtype, turns=res.num_turns, structured=res.structured_output,
        usage={k: x for k, x in (res.usage or {}).items() if "tokens" in k})

In [ ]:
sample = next(c["text"] for c in chunks if len(c["text"]) > 1500)
for model in ["haiku", "sonnet"]:
    try:
        ans, meta = await ask(f"Summarize this excerpt in one sentence:\n\n{sample}", model=model)
        print(model, "->", (ans or "")[:300])
        record("sdk", **{f"{model}_summary": meta})
        times = []
        for _ in range(3):
            _, m2 = await ask("Reply with the single word: ok", model=model)
            times.append((m2["wall_s"], m2.get("api_s")))
        record("sdk", **{f"{model}_tiny_calls_wall_api_s": times})
    except Exception as e:
        record("sdk", **{f"{model}_exception": repr(e)[:400]})
        print("\n".join(STDERR[-30:]))

In [ ]:
ROUTE_SCHEMA = {"type": "object",
                "properties": {"route": {"type": "string", "enum": ["direct", "simple", "complex"]},
                               "reason": {"type": "string"}},
                "required": ["route", "reason"], "additionalProperties": False}
Q = ("Compare the post-merger HHI figures for each overlapping regional market against the Merger "
     "Guidelines thresholds and recommend a remedy strategy.")
for mt in [1, 3]:
    try:
        _, meta = await ask(f"Classify this query's complexity for a retrieval system.\nQuery: {Q}",
                            model="haiku", max_turns=mt,
                            output_format={"type": "json_schema", "schema": ROUTE_SCHEMA})
        record("sdk", **{f"structured_max_turns_{mt}": {k: meta.get(k) for k in
                                                         ["structured", "turns", "subtype", "is_error", "wall_s"]}})
    except Exception as e:
        record("sdk", **{f"structured_max_turns_{mt}": repr(e)[:400]})

In [ ]:
async def bench_concurrency(n):
    t0 = time.perf_counter()
    rs = await asyncio.gather(*[ask(f"Reply with the number {i} only.", model="haiku") for i in range(n)],
                              return_exceptions=True)
    ok = sum(1 for r in rs if not isinstance(r, Exception) and r[1].get("is_error") is False)
    errs = [repr(r)[:150] if isinstance(r, Exception) else r[1].get("subtype")
            for r in rs if isinstance(r, Exception) or r[1].get("is_error")]
    return dict(n=n, ok=ok, wall_s=round(time.perf_counter() - t0, 1), errors=errs[:3])

for n in [4, 8]:
    record("sdk", **{f"concurrency_{n}": await bench_concurrency(n)})

In [ ]:
# Optional: the API-key fallback path (only runs if you added the ANTHROPIC_API_KEY secret)
if API_KEY:
    import anthropic
    try:
        t0 = time.perf_counter()
        r = anthropic.Anthropic(api_key=API_KEY).messages.create(
            model="claude-haiku-5-5", max_tokens=20, messages=[{"role": "user", "content": "Reply: ok"}])
        record("fallback_api", ok=True, wall_s=round(time.perf_counter() - t0, 2), text=r.content[0].text)
    except Exception as e:
        record("fallback_api", ok=False, error=repr(e)[:300])
else:
    print("No ANTHROPIC_API_KEY secret; skipping the fallback test.")

## 3. Retrieval models on the GPU: BGE-M3 + FAISS + BM25 + bge-reranker-v2-m3
We also preview **why hybrid + rerank matters**. For a query whose answer contains the exact figure "3,552", we look at where the chunk holding that figure ranks under dense, BM25 and reranked retrieval.

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer, CrossEncoder
torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
gpu_report("before_step3")

t0 = time.perf_counter()
emb = SentenceTransformer("BAAI/bge-m3", device="cuda", model_kwargs={DTYPE_KW: torch.float16})
emb.max_seq_length = 1024
record("embed", load_s=round(time.perf_counter() - t0, 1))

texts = [c["text"] for c in chunks]
for bs in [16, 32, 64]:
    torch.cuda.synchronize(); t0 = time.perf_counter()
    emb.encode(texts[:512], batch_size=bs, normalize_embeddings=True, show_progress_bar=False)
    torch.cuda.synchronize()
    record("embed", **{f"bs{bs}_chunks_per_s": round(512 / (time.perf_counter() - t0), 1)})

t0 = time.perf_counter()
E = emb.encode(texts, batch_size=32, normalize_embeddings=True, show_progress_bar=True).astype("float32")
record("embed", full_corpus_s=round(time.perf_counter() - t0, 1), n=len(texts), dim=E.shape[1], peak_vram_gb=peak_gb())

In [ ]:
import faiss, bm25s
Q_HHI = "post-merger HHI and HHI delta for flame retardants in the South Central region"
gold = [i for i, t in enumerate(texts) if "3,552" in t]
print("chunks containing '3,552':", gold)

index = faiss.IndexFlatIP(E.shape[1]); index.add(E)
t0 = time.perf_counter()
qv = emb.encode([Q_HHI], normalize_embeddings=True).astype("float32")
D, I = index.search(qv, 50)
dense_ms = round((time.perf_counter() - t0) * 1000, 1)

t0 = time.perf_counter()
bm = bm25s.BM25(); bm.index(bm25s.tokenize(texts, stopwords="en", show_progress=False), show_progress=False)
bm_index_s = round(time.perf_counter() - t0, 2)
t0 = time.perf_counter()
bres, bscores = bm.retrieve(bm25s.tokenize([Q_HHI], stopwords="en", show_progress=False), k=50,
                            show_progress=False, backend_selection="numpy")
gpu_report("after_bm25")   # should stay roughly unchanged now that JAX is kept off the GPU
bm_ms = round((time.perf_counter() - t0) * 1000, 1)

def rank_of(ids, gold):
    ids = list(map(int, ids))
    r = [ids.index(g) + 1 for g in gold if g in ids]
    return min(r) if r else None

record("index", dense_query_ms=dense_ms, bm25_index_s=bm_index_s, bm25_query_ms=bm_ms,
       gold_rank_dense=rank_of(I[0], gold), gold_rank_bm25=rank_of(bres[0], gold))

In [ ]:
def load_ce():
    name = "BAAI/bge-reranker-v2-m3"
    try:
        return CrossEncoder(name, device="cuda", max_length=1024, model_kwargs={DTYPE_KW: torch.float16})
    except TypeError:
        ce = CrossEncoder(name, device="cuda", max_length=1024); ce.model.half(); return ce

t0 = time.perf_counter(); ce = load_ce(); record("rerank", load_s=round(time.perf_counter() - t0, 1))

# RRF fusion of the dense + BM25 candidates (k=60), then rerank the top 30
rrf = collections.defaultdict(float)
for ranking in (list(map(int, I[0])), list(map(int, bres[0]))):
    for r, idx in enumerate(ranking):
        rrf[idx] += 1 / (60 + r + 1)
cands = [i for i, _ in sorted(rrf.items(), key=lambda kv: -kv[1])][:30]

torch.cuda.synchronize(); t0 = time.perf_counter()
scores = ce.predict([(Q_HHI, texts[i]) for i in cands], batch_size=16, show_progress_bar=False)
torch.cuda.synchronize(); rerank_ms = round((time.perf_counter() - t0) * 1000)
reranked = [cands[j] for j in np.argsort(-scores)]

# CRAG-style strip scoring: split the top-8 chunks into sentences and score each one
import re
strips = [s for i in reranked[:8] for s in re.split(r"(?<=[.;:])\s+|\n+", texts[i]) if len(s.split()) >= 4]
torch.cuda.synchronize(); t0 = time.perf_counter()
sscores = ce.predict([(Q_HHI, s) for s in strips], batch_size=64, show_progress_bar=False)
torch.cuda.synchronize(); strip_ms = round((time.perf_counter() - t0) * 1000)

record("rerank", rerank_30_ms=rerank_ms, gold_rank_rrf=rank_of(cands, gold), gold_rank_reranked=rank_of(reranked, gold),
       top_scores=[round(float(s), 3) for s in sorted(scores, reverse=True)[:8]],
       score_quantiles=[round(float(q), 3) for q in np.quantile(scores, [0.1, 0.5, 0.9])],
       n_strips=len(strips), strip_scoring_ms=strip_ms,
       strip_top=[(round(float(sscores[j]), 3), strips[j][:120]) for j in np.argsort(-sscores)[:3]],
       peak_vram_gb=peak_gb())
print("\nTop reranked chunk:\n", texts[reranked[0]][:800])

In [ ]:
del emb, ce; gc.collect(); torch.cuda.empty_cache()
gpu_report("after_step3_free")

## 4. Observability: Phoenix + a traced toy LangGraph that calls Claude
The graph has the same shape as the real one: router → retrieve (manual RETRIEVER span) → generate (manual LLM span around the SDK call). The SDK isn't auto-instrumented, so we wrap it by hand. LangGraph nodes are auto-instrumented through OpenInference.

In [ ]:
import phoenix as px
from phoenix.otel import register
session = px.launch_app()
tp = register(project_name="hrag-smoke", endpoint="http://127.0.0.1:6006/v1/traces",
              protocol="http/protobuf", auto_instrument=True, batch=False)
tracer = tp.get_tracer("hrag")
from google.colab.output import eval_js
print("Phoenix UI →", eval_js("google.colab.kernel.proxyPort(6006)"))

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class S(TypedDict, total=False):
    q: str; route: str; docs: list; answer: str

def router(s):
    return {"route": "simple" if len(s["q"]) < 80 else "complex"}

def retrieve(s):
    with tracer.start_as_current_span("hybrid_retrieve", attributes={
            "openinference.span.kind": "RETRIEVER", "input.value": s["q"]}) as sp:
        docs = [texts[i][:300] for i in reranked[:3]]
        for i, d in enumerate(docs):
            sp.set_attribute(f"retrieval.documents.{i}.document.content", d)
        return {"docs": docs}

async def generate(s):
    prompt = "Answer from the context only.\n\n" + "\n---\n".join(s["docs"]) + f"\n\nQuestion: {s['q']}"
    with tracer.start_as_current_span("claude_generate", attributes={
            "openinference.span.kind": "LLM", "llm.model_name": "haiku", "input.value": prompt}) as sp:
        ans, meta = await ask(prompt, model="haiku")
        sp.set_attribute("output.value", ans or "")
        for k, x in meta.get("usage", {}).items():
            sp.set_attribute(f"llm.usage.{k}", x)
        return {"answer": ans}

g = StateGraph(S)
g.add_node("router", router); g.add_node("retrieve", retrieve); g.add_node("generate", generate)
g.add_edge(START, "router")
g.add_conditional_edges("router", lambda s: s["route"], {"simple": "retrieve", "complex": "retrieve"})
g.add_edge("retrieve", "generate"); g.add_edge("generate", END)
graph = g.compile()

t0 = time.perf_counter()
out = await graph.ainvoke({"q": Q_HHI})
print(out["answer"])
record("phoenix", graph_run_s=round(time.perf_counter() - t0, 1), answer_preview=(out.get("answer") or "")[:200])

In [ ]:
time.sleep(3)
try:
    from phoenix.client import Client
    df = Client(base_url="http://127.0.0.1:6006").spans.get_spans_dataframe(project_identifier="hrag-smoke")
    record("phoenix", n_spans=len(df), span_names=sorted(df["name"].unique().tolist())[:20])
except Exception as e:
    record("phoenix", span_check_error=repr(e)[:300])
print("Also open the Phoenix UI link above and check that you see the trace tree.")

## 5. Contextual Retrieval with a small local LLM
Anthropic's prompt: the **whole document** followed by the **chunk**, asking for a 50–100-token context. All chunks of one document share the same long prefix, so a **prefix cache** (vLLM `enable_prefix_caching`) reuses the document's KV cache across them. That's the local analogue of Anthropic's prompt caching.

- **5a.** HF transformers in the main env, a small sample. This is the fallback that will definitely run.
- **5b.** vLLM in an **isolated venv** run as a **subprocess**, so it can't break this notebook's torch and the GPU memory is freed when the process exits. vLLM support on the T4 (SM 7.5) is uncertain, which is exactly why we test it.

**Running step 5 on its own (e.g. after a restart):** run the install cell, the env cell, and the step 1 cells (data, ~1 min), then jump here. Steps 2–4 aren't needed.

In [ ]:
DOC_PROMPT = "<document>\n{doc}\n</document>"
CHUNK_PROMPT = ("Here is the chunk we want to situate within the whole document\n<chunk>\n{chunk}\n</chunk>\n"
                "Please give a short succinct context to situate this chunk within the overall document for the "
                "purposes of improving search retrieval of the chunk. Answer only with the succinct context and nothing else.")
MAX_DOC_TOK = 12000
SMALL_LLM = "Qwen/Qwen2.5-3B-Instruct"   # candidate alternative: "Qwen/Qwen3-4B-Instruct-2507"

# ~200 chunks grouped by document (so the prefix cache gets reuse), docs capped at MAX_DOC_TOK
by_doc = collections.defaultdict(list)
for c in chunks:
    by_doc[(c["task"], c["doc"])].append(c["text"])
doc_text = {(d["task"], d["doc"]): d["text"] for d in docs if d["n_tok"] <= MAX_DOC_TOK}
samples = []
for key, cs in by_doc.items():
    if key in doc_text and len(cs) >= 3:
        samples += [(doc_text[key], c) for c in cs[:8]]
    if len(samples) >= 200:
        break
print(len(samples), "samples from", len({s[0] for s in samples}), "documents")

In [ ]:
from transformers import AutoModelForCausalLM
gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
g = gpu_report("before_5a")
if g["free_gb"] < 8:
    raise RuntimeError(f"Only {g['free_gb']} GB free but the 3B model needs ~6.2 GB of weights plus activations. "
                       "Something else holds GPU memory (see gpu_procs above). Restart the session and run: "
                       "install → env → step 1 → step 5.")
t0 = time.perf_counter()
stok = AutoTokenizer.from_pretrained(SMALL_LLM); stok.padding_side = "left"
slm = AutoModelForCausalLM.from_pretrained(SMALL_LLM, device_map="cuda", attn_implementation="sdpa",
                                           **{DTYPE_KW: torch.float16})
record("ctx_hf", model=SMALL_LLM, load_s=round(time.perf_counter() - t0, 1))

def build(doc, chunk):
    return stok.apply_chat_template([{"role": "user", "content": DOC_PROMPT.format(doc=doc) + "\n\n" + CHUNK_PROMPT.format(chunk=chunk)}],
                                    tokenize=False, add_generation_prompt=True)

N_HF, BS = 16, 4
outs = []; t0 = time.perf_counter()
try:
    for i in range(0, N_HF, BS):
        enc = stok([build(d, c) for d, c in samples[i:i + BS]], return_tensors="pt", padding=True).to("cuda")
        with torch.no_grad():
            gen = slm.generate(**enc, max_new_tokens=100, do_sample=False)
        outs += stok.batch_decode(gen[:, enc.input_ids.shape[1]:], skip_special_tokens=True)
    per_chunk = (time.perf_counter() - t0) / N_HF
    record("ctx_hf", s_per_chunk=round(per_chunk, 2),
           est_full_corpus_min=round(per_chunk * len(chunks) / 60, 1), peak_vram_gb=peak_gb(),
           examples=[o.strip()[:300] for o in outs[:3]])
except torch.cuda.OutOfMemoryError as e:
    record("ctx_hf", oom=True, batch=BS, error=str(e)[:200])
del slm; gc.collect(); torch.cuda.empty_cache()
gpu_report("after_5a_free")   # vLLM (5b) needs this memory back: its gpu_memory_utilization=0.85 is a share of the WHOLE GPU

In [ ]:
t0 = time.perf_counter()
r = subprocess.run("pip -q install uv && uv venv -q /content/vllm_env --python 3.12 && "
                   "uv pip install -q --python /content/vllm_env/bin/python vllm",
                   shell=True, capture_output=True, text=True)
record("ctx_vllm", install_s=round(time.perf_counter() - t0, 1), install_ok=r.returncode == 0,
       install_err=r.stderr[-600:] if r.returncode else None)

In [ ]:
VLLM_SCRIPT = r"""
import json, sys, time
from vllm import LLM, SamplingParams
cfg = json.load(open(sys.argv[1]))
t0 = time.perf_counter()
llm = LLM(model=cfg["model"], dtype="half", max_model_len=cfg["max_len"],
          gpu_memory_utilization=0.85, enable_prefix_caching=True)
load_s = time.perf_counter() - t0
msgs = [[{"role": "user", "content": cfg["doc_prompt"].format(doc=d) + "\n\n" + cfg["chunk_prompt"].format(chunk=c)}]
        for d, c in cfg["samples"]]
t0 = time.perf_counter()
outs = llm.chat(msgs, SamplingParams(temperature=0, max_tokens=100), use_tqdm=False)
gen_s = time.perf_counter() - t0
json.dump(dict(load_s=load_s, gen_s=gen_s, n=len(msgs), examples=[o.outputs[0].text for o in outs[:3]]),
          open(cfg["out"], "w"))
"""
open("/content/bench_vllm.py", "w").write(VLLM_SCRIPT)

def run_vllm(model):
    cfg = dict(model=model, max_len=16384, doc_prompt=DOC_PROMPT, chunk_prompt=CHUNK_PROMPT,
               samples=samples, out="/content/vllm_out.json")
    json.dump(cfg, open("/content/vllm_cfg.json", "w"))
    t0 = time.perf_counter()
    try:
        p = subprocess.run(["/content/vllm_env/bin/python", "/content/bench_vllm.py", "/content/vllm_cfg.json"],
                           capture_output=True, text=True, timeout=1800)
    except subprocess.TimeoutExpired:
        return dict(model=model, ok=False, error="timeout 30 min")
    log = p.stdout + p.stderr
    backend = [l[-200:] for l in log.splitlines() if "backend" in l.lower()][:4]
    if p.returncode != 0:
        return dict(model=model, ok=False, total_s=round(time.perf_counter() - t0, 1), backend_log=backend, error=log[-1500:])
    o = json.load(open("/content/vllm_out.json"))
    return dict(model=model, ok=True, load_s=round(o["load_s"], 1), n=o["n"],
                s_per_chunk=round(o["gen_s"] / o["n"], 3),
                est_full_corpus_min=round(o["gen_s"] / o["n"] * len(chunks) / 60, 1),
                backend_log=backend, examples=[e.strip()[:300] for e in o["examples"]])

if RESULTS.get("ctx_vllm", {}).get("install_ok"):
    record("ctx_vllm", run_1=run_vllm(SMALL_LLM))

In [ ]:
# Optional: try a second small model if the first one worked and there's time left
TRY_SECOND = False
if TRY_SECOND and RESULTS.get("ctx_vllm", {}).get("run_1", {}).get("ok"):
    record("ctx_vllm", run_2=run_vllm("Qwen/Qwen3-4B-Instruct-2507"))

### 5c. Per-document contextualization (the redesign)
Per-chunk prompts resend the whole document (~8k tokens) for every ~60-token context, and that is what made 5b take ~3 h. Here the document is sent **once**, with chunk markers `[[c1]] … [[cN]]` inline, and the model writes one line per chunk (`[c1] …`). The document is read once per document instead of once per chunk, the same effect prompt caching is supposed to give.

We measure **speed** and **coverage**: the share of chunks that come back with a parseable context line. Missed chunks fall back to the deterministic header in the real build.

**If your step 5 runtime is still alive**, just run this cell. If it was recycled, run install → env → step 1 → the first step 5 cell (samples) → the vLLM install cell → this cell.

In [ ]:
PERDOC_INSTR = ("The document above has been split into consecutive chunks; each chunk starts with a marker like [[c1]], [[c2]], ...\n"
                "For EACH chunk, write one short, succinct context (1-2 sentences, at most 50 words) that situates the chunk "
                "within the overall document, to improve search retrieval of the chunk: name the parties or matter and "
                "what that part of the document is about.\n"
                "Answer with exactly one line per chunk, in order, formatted as:\n[c1] <context>\n[c2] <context>\n...\n"
                "Output nothing else.")

def marked_doc(chunk_texts):
    return "\n".join(f"[[c{i}]] {t}" for i, t in enumerate(chunk_texts, 1))

# 20 docs (<= MAX_DOC_TOK) spread across the size range
eligible = sorted([k for k in by_doc if k in doc_text], key=lambda k: len(by_doc[k]))
N_DOCS = 20
pick = eligible[::max(1, len(eligible) // N_DOCS)][:N_DOCS]
perdoc = [dict(key=list(k), n=len(by_doc[k]),
               prompt=f"<document>\n{marked_doc(by_doc[k])}\n</document>\n\n{PERDOC_INSTR}") for k in pick]
max_n = max(d["n"] for d in perdoc)
print(len(perdoc), "docs,", sum(d["n"] for d in perdoc), "chunks, max chunks/doc =", max_n)

PERDOC_SCRIPT = r"""
import json, re, sys, time
from vllm import LLM, SamplingParams
cfg = json.load(open(sys.argv[1]))
t0 = time.perf_counter()
llm = LLM(model=cfg["model"], dtype="half", max_model_len=cfg["max_len"], gpu_memory_utilization=0.85)
load_s = time.perf_counter() - t0
msgs = [[{"role": "user", "content": d["prompt"]}] for d in cfg["docs"]]
sp = SamplingParams(temperature=0, max_tokens=cfg["max_new"])
t0 = time.perf_counter()
outs = llm.chat(msgs, sp, use_tqdm=False)
gen_s = time.perf_counter() - t0
res = []
for d, o in zip(cfg["docs"], outs):
    text = o.outputs[0].text
    got = {int(m.group(1)): m.group(2).strip() for m in re.finditer(r"^\[c(\d+)\]\s*(.+)$", text, re.M)}
    res.append(dict(key=d["key"], n=d["n"], parsed=sum(1 for i in range(1, d["n"] + 1) if i in got),
                    prompt_tokens=len(o.prompt_token_ids), out_tokens=len(o.outputs[0].token_ids),
                    finish=o.outputs[0].finish_reason, contexts=[got.get(i) for i in range(1, d["n"] + 1)],
                    raw_head=text[:300]))
json.dump(dict(load_s=load_s, gen_s=gen_s, docs=res), open(cfg["out"], "w"))
"""
open("/content/bench_vllm_perdoc.py", "w").write(PERDOC_SCRIPT)
json.dump(dict(model=SMALL_LLM, max_len=16384, max_new=90 * max_n + 50, docs=perdoc, out="/content/vllm_perdoc_out.json"),
          open("/content/vllm_perdoc_cfg.json", "w"))

t0 = time.perf_counter()
try:
    p = subprocess.run(["/content/vllm_env/bin/python", "/content/bench_vllm_perdoc.py", "/content/vllm_perdoc_cfg.json"],
                       capture_output=True, text=True, timeout=1800)
    if p.returncode != 0:
        record("ctx_perdoc", ok=False, error=(p.stdout + p.stderr)[-1500:])
    else:
        o = json.load(open("/content/vllm_perdoc_out.json"))
        D = o["docs"]
        n_chunks = sum(d["n"] for d in D); parsed = sum(d["parsed"] for d in D)
        s_per_chunk = o["gen_s"] / n_chunks
        record("ctx_perdoc", ok=True, model=SMALL_LLM, load_s=round(o["load_s"], 1), n_docs=len(D), n_chunks=n_chunks,
               gen_s=round(o["gen_s"], 1), s_per_chunk=round(s_per_chunk, 3),
               est_full_corpus_min=round(s_per_chunk * len(chunks) / 60, 1),
               coverage=round(parsed / n_chunks, 3),
               docs_full_coverage=sum(d["parsed"] == d["n"] for d in D),
               hit_max_tokens=sum(d["finish"] == "length" for d in D),
               prompt_tokens=sum(d["prompt_tokens"] for d in D), out_tokens=sum(d["out_tokens"] for d in D),
               examples=[(d["key"][1], i + 1, c[:250]) for d in D[:3] for i, c in enumerate(d["contexts"][:2]) if c],
               worst_doc=min(D, key=lambda d: d["parsed"] / d["n"])["raw_head"])
except subprocess.TimeoutExpired:
    record("ctx_perdoc", ok=False, error="timeout 30 min")
print("wall incl. model load:", round(time.perf_counter() - t0, 1), "s")

## 6. Copy everything below back to Claude

In [ ]:
print(json.dumps(RESULTS, indent=1, default=str))